# 102. Binary Tree Level Order Traversal
**Difficulty:** 🟡 Medium · **Topic:** Tree · **LeetCode:** https://leetcode.com/problems/binary-tree-level-order-traversal/

## 💡 Concepts

**Core concept(s):** **BFS with a queue** (process one full level at a time); or DFS that tags each value with its depth.

**Why it applies here:** "Level by level" is exactly what a queue gives you: hold one level, expand it to produce the next.

**Key intuition:** Take everything currently in the queue — that's one level — record it, and push the next level's nodes.

---

### 📚 What is a Binary Tree?
A **binary tree** is nodes in a branching shape: each node holds a value and up to two children (**left**, **right**). The top is the **root**; childless nodes are **leaves**; **height** is the longest root-to-leaf path.
- **In Python:** a small `TreeNode` class with `.val`, `.left`, `.right`.

### 📚 What is BFS (Breadth-First Search) / a Queue?
**BFS** explores level by level using a **queue** (a first-in-first-out line): take a node, push its children to the back, repeat.
- **Complexity:** **O(n)** time; the queue can hold up to one full level.
- **In Python:** `collections.deque` (`append` to add, `popleft` to take from the front).

### 📚 What is DFS (Depth-First Search) / Recursion?
**DFS** dives down one branch as far as possible, then backtracks. It's usually written with **recursion** — a function that calls itself on each child.
- **Complexity:** visits each node once → **O(n)** time; uses call-stack space up to the tree's **height**.

---

**Prerequisite knowledge:**
- A queue (`deque`).
- Recursion with a depth argument.

## 📝 Problem

Return the node values grouped by level, top to bottom, left to right.

**Example**
```
[3,9,20,None,None,15,7] -> [[3],[9,20],[15,7]]
```

> Two approaches, both `O(n)`: BFS (natural) and DFS (tag by depth).

In [ ]:
from typing import Optional, List
from collections import deque

class TreeNode:
    """A single node of a binary tree: a value plus links to up to two children."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val                     # the number stored at this node
        self.left = left                   # the left child (or None)
        self.right = right                 # the right child (or None)

def build_tree(values):
    """Build a tree from a level-order list, LeetCode style (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()                 # the parent we're attaching children to
        if i < len(values):                # attach the left child (if present)
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):                # attach the right child (if present)
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def build_balanced(n):
    """Balanced BST holding 1..n (height ~log n) — used by the benchmark."""
    def helper(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2               # middle value becomes the subtree's root
        node = TreeNode(mid)
        node.left = helper(lo, mid - 1)    # smaller values go left
        node.right = helper(mid + 1, hi)   # larger values go right
        return node
    return helper(1, n)

def preorder(root):
    """Collect values in preorder: node, then left, then right."""
    out = []
    def go(n):
        if not n: return
        out.append(n.val); go(n.left); go(n.right)
    go(root); return out

def inorder(root):
    """Collect values in inorder: left, then node, then right (sorted for a BST)."""
    out = []
    def go(n):
        if not n: return
        go(n.left); out.append(n.val); go(n.right)
    go(root); return out

def same_shape(a, b):
    """True if two trees have identical shape and values."""
    if not a and not b: return True        # both empty -> match
    if not a or not b or a.val != b.val: return False  # one empty, or values differ
    return same_shape(a.left, b.left) and same_shape(a.right, b.right)

### Approach 1 — BFS with a Queue (natural)

**Idea:** The queue always holds exactly one level; empty it into a list, pushing children for the next level.

**Time:** `O(n)`. **Space:** `O(n)`.

In [ ]:
def level_order_bfs(root: Optional[TreeNode]) -> List[List[int]]:
    res = []
    if not root:
        return res
    q = deque([root])
    while q:
        level = []                         # values collected for the current level
        for _ in range(len(q)):            # snapshot: current queue = exactly one level
            node = q.popleft()
            level.append(node.val)
            if node.left:  q.append(node.left)   # push the next level's nodes
            if node.right: q.append(node.right)
        res.append(level)                  # store this finished level
    return res

### Approach 2 — DFS Tagged by Depth

**Idea:** Recurse with a depth counter; append each value to the list for its depth.

**Time:** `O(n)`. **Space:** `O(h)` (+ output).

In [ ]:
def level_order_dfs(root: Optional[TreeNode]) -> List[List[int]]:
    res = []
    def dfs(node, depth):
        if not node:
            return
        if depth == len(res):              # first time we reach this depth...
            res.append([])                 # ...create its (empty) list
        res[depth].append(node.val)        # add this node's value to its depth's list
        dfs(node.left, depth + 1)          # children live one level deeper
        dfs(node.right, depth + 1)
    dfs(root, 0)
    return res

In [ ]:
# Correctness check
tests = [([3,9,20,None,None,15,7],[[3],[9,20],[15,7]]), ([1],[[1]]), ([],[])]
for vals, exp in tests:
    root = build_tree(vals)
    a, b = level_order_bfs(root), level_order_dfs(root)
    print(f"{vals} -> {a}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on trees of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n²)`       | ≈ **4×** |

We use **balanced** trees (height ~log n) so deep recursion stays safe while every node is still visited.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    return (build_balanced(n),)
solutions = {
    "bfs O(n)": level_order_bfs,
    "dfs O(n)": level_order_dfs,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **BFS = level-by-level:** whenever grouping or distance-by-levels matters, reach for a queue.
- **The "snapshot the queue size" trick:** freezing `len(q)` cleanly separates one level from the next.
- **Signal:** "level order", "by depth", "row by row", "shortest path in an unweighted graph".
- **Related problems:** Zigzag Level Order, Right Side View, Minimum Depth.
- **Common pitfalls:** (1) reading `len(q)` inside the loop after it changed; (2) forgetting the empty-tree case.